# SpeakerForge Phase 2 — CSM-1B LoRA 训练 (Colab T4)

**前提：**
1. 本地已运行 `stage1`，生成 `sesame/data/Akinokoe_vB/`
   - stage1 从 `../speakerforge/dataset/Akinokoe_versionB/` 读取数据（Phase 1 stage7 输出）
   - stage1 将音频以 WAV bytes 嵌入 Parquet，数据集完全自包含，可移植到 Colab
2. 已将 `sesame/data/Akinokoe_vB/` 复制到 Google Drive（如 `My Drive/SpeakerForge/sesame/data/Akinokoe_vB/`）
3. HuggingFace read token 已添加到 Colab Secrets（key 名：`HF_TOKEN`）

**数据集格式（以 notebook preprocess_example 为依据）：**
- `audio`：HF Audio feature，24kHz，float32
- `text`：str 转录文本
- `source`：str 说话人ID（单说话人默认 "0"，在 Cell 4 自动添加）

训练完成后，adapter 保存到 Drive，本地直接通过 G: 盘访问，然后运行 `stage3`。

In [ ]:
# ── Cell 1: 安装依赖 ──────────────────────────────────────────────────────────
%%capture
import os, re
import torch
v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.52.3
!pip install --no-deps trl==0.22.2
!pip install torchcodec "datasets>=3.4.1,<4.0.0"

In [ ]:
# ── Cell 2: 挂载 Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# 修改为你 Drive 中实际的路径
DRIVE_ROOT = "/content/drive/MyDrive/SpeakerForge/sesame"
DATA_DIR   = f"{DRIVE_ROOT}/data/Akinokoe_vB"      # stage1 输出
OUTPUT_DIR = f"{DRIVE_ROOT}/models/Akinokoe"        # adapter 保存位置

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Data:   {DATA_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Exists: {os.path.exists(DATA_DIR)}")

In [ ]:
# ── Cell 3: 加载模型 + LoRA ───────────────────────────────────────────────────
from unsloth import FastModel
from transformers import CsmForConditionalGeneration
import inspect
import transformers.models.csm.modeling_csm as _csm_mod

MODEL_NAME = "unsloth/csm-1b"

model, processor = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    dtype=None,
    auto_model=CsmForConditionalGeneration,
    load_in_4bit=False,
)

# ── Patch BEFORE get_peft_model ───────────────────────────────────────────────
# Unsloth GC 在 get_peft_model 时把 forward 存入 closure；必须先 patch。
# 注意：不能用 functools.wraps，否则 inspect.unwrap() 会绕过 wrapper 直接调原函数。
_target_cls = None
for _name in dir(_csm_mod):
    _obj = getattr(_csm_mod, _name)
    if isinstance(_obj, type) and hasattr(_obj, 'forward'):
        _params = list(inspect.signature(_obj.forward).parameters)
        if 'backbone_last_hidden_state' in _params and 'inputs_embeds' in _params:
            _target_cls = _obj
            print(f"Patching: {_name}")
            break

if _target_cls is None:
    print("WARNING: depth decoder class not found — patch not applied")
else:
    _orig_depth_fwd = _target_cls.forward
    def _patched_depth_fwd(self, *args, **kwargs):
        if 'inputs_embeds' in kwargs:
            if kwargs['inputs_embeds'] is not None:
                kwargs['inputs_embeds'] = kwargs['inputs_embeds'].clone()
        elif len(args) > 5 and args[5] is not None:
            args = args[:5] + (args[5].clone(),) + args[6:]
        return _orig_depth_fwd(self, *args, **kwargs)
    _target_cls.forward = _patched_depth_fwd
    print("Patch applied.")

# ── LoRA — 与官方对齐 ─────────────────────────────────────────────────────────
model = FastModel.get_peft_model(
    model,
    r=32,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)
print("Model loaded.")

In [ ]:
# ── Cell 4: 数据集预处理 ──────────────────────────────────────────────────────
# 与官方对齐：显式用 AutoProcessor（官方 Cell 10 会覆盖 FastModel 返回的 processor）
import torch
from datasets import load_from_disk, Audio
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_NAME)

raw_ds = load_from_disk(DATA_DIR)
raw_ds = raw_ds.cast_column("audio", Audio(sampling_rate=24000))
if "source" not in raw_ds.column_names:
    raw_ds = raw_ds.add_column("source", ["0"] * len(raw_ds))
print(f"Raw dataset: {len(raw_ds)} samples")

def preprocess_example(example):
    conversation = [{
        "role": example["source"],
        "content": [
            {"type": "text",  "text":  example["text"]},
            {"type": "audio", "path":  example["audio"]["array"]},
        ],
    }]
    try:
        model_inputs = processor.apply_chat_template(
            conversation, tokenize=True, return_dict=True, output_labels=True,
            text_kwargs={"padding": "max_length", "max_length": 256,
                         "pad_to_multiple_of": 8, "padding_side": "right"},
            audio_kwargs={"sampling_rate": 24_000, "max_length": 240001,
                          "padding": "max_length"},
            common_kwargs={"return_tensors": "pt"},
        )
    except Exception as e:
        print(f"  Skip '{example['text'][:40]}': {e}")
        return None
    required = ["input_ids", "attention_mask", "labels", "input_values", "input_values_cutoffs"]
    result = {k: model_inputs[k][0] for k in required if k in model_inputs}
    if len(result) < len(required):
        return None
    return result

processed_ds = raw_ds.map(
    preprocess_example,
    remove_columns=raw_ds.column_names,
    desc="Preprocessing",
)
processed_ds = processed_ds.filter(lambda ex: ex is not None)
print(f"Processed: {len(processed_ds)} samples")

In [ ]:
# ── Cell 5: 训练 ──────────────────────────────────────────────────────────────
from transformers import TrainingArguments, Trainer
from unsloth import is_bfloat16_supported

MAX_STEPS = 120

trainer = Trainer(
    model=model,
    train_dataset=processed_ds,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=OUTPUT_DIR,
        report_to="none",
        remove_unused_columns=False,  # CSM audio columns not in standard forward sig
    ),
)

gpu_stats = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu_stats.name}, {gpu_stats.total_memory/1024**3:.1f} GB")

trainer_stats = trainer.train()
print(f"\nTraining done: {trainer_stats.metrics['train_runtime']:.0f}s  "
      f"({trainer_stats.metrics['train_runtime']/60:.1f} min)")

In [ ]:
# ── Cell 6: 保存 adapter 到 Drive ────────────────────────────────────────────
adapter_path = f"{OUTPUT_DIR}/lora_adapter"
model.save_pretrained(adapter_path)
processor.save_pretrained(adapter_path)
print(f"Adapter saved → {adapter_path}")
print("\n本地 G: 盘路径（可直接用于 stage3）：")
print(adapter_path.replace("/content/drive/MyDrive", "G:").replace("/", "\\"))

In [ ]:
# ── Cell 7: 快速推理验证（可选）─────────────────────────────────────────────
import inspect
import soundfile as sf
from IPython.display import Audio, display
from transformers.generation import utils as _gen_utils
from transformers.models.csm.modeling_csm import CsmDepthDecoderForCausalLM

# 与官方对齐：不调用 FastModel.for_inference()，output_audio=True，传全量 **inputs

# Patch 1: idempotent — backbone_last_hidden_state 签名验证
# 本 session Cell 3 已运行（无 functools.wraps），签名丢失，需此 patch
# 下次 session Cell 3 有 functools.wraps 后此 patch 自动跳过（有 orignal 签名可见）
if not hasattr(_gen_utils.GenerationMixin, '_csm_orig_validate_model_kwargs'):
    _gen_utils.GenerationMixin._csm_orig_validate_model_kwargs = (
        _gen_utils.GenerationMixin._validate_model_kwargs
    )
def _global_validate(self, model_kwargs):
    model_kwargs.pop('backbone_last_hidden_state', None)
    return _gen_utils.GenerationMixin._csm_orig_validate_model_kwargs(self, model_kwargs)
_gen_utils.GenerationMixin._validate_model_kwargs = _global_validate

# Patch 2: idempotent — KeyError: 'position_ids' in prepare_inputs_for_generation
for _dname, _dmod in model.named_modules():
    if not hasattr(_dmod, 'generate'):
        continue
    _actual_cls = type(_dmod)
    if 'prepare_inputs_for_generation' not in vars(_actual_cls):
        continue
    if getattr(_actual_cls, '_csm_prep_patched', False):
        continue
    try:
        _src = inspect.getsource(vars(_actual_cls)['prepare_inputs_for_generation'])
        if 'position_ids' not in _src:
            continue
    except Exception:
        pass
    _orig_prep = vars(_actual_cls)['prepare_inputs_for_generation']
    def _make_safe_prep(orig):
        def _safe_prep(self, input_ids, *args, **kwargs):
            try:
                return orig(self, input_ids, *args, **kwargs)
            except KeyError as _e:
                if 'position_ids' not in str(_e):
                    raise
                for _base in type(self).__mro__[1:]:
                    if 'prepare_inputs_for_generation' in vars(_base):
                        _mi = vars(_base)['prepare_inputs_for_generation'](
                            self, input_ids, *args, **kwargs)
                        _mi.pop('position_ids', None)
                        return _mi
                raise
        return _safe_prep
    _actual_cls.prepare_inputs_for_generation = _make_safe_prep(_orig_prep)
    _actual_cls._csm_prep_patched = True
    print(f"Patched prepare_inputs_for_generation: {_actual_cls.__name__} ({_dname})")

# Patch 3: idempotent — non-int logits_to_keep → empty cache_position → CsmCodebooksHead fails
# 直接 target CsmDepthDecoderForCausalLM，不依赖 signature 扫描（Cell 3 的 wrapper 会隐藏签名）
if not getattr(CsmDepthDecoderForCausalLM, '_csm_logits_to_keep_patched', False):
    _orig_dd_fwd = CsmDepthDecoderForCausalLM.forward
    def _make_ltk_fix(orig):
        def _ltk_fixed_fwd(self, *args, **kwargs):
            if 'logits_to_keep' in kwargs and not isinstance(kwargs['logits_to_keep'], int):
                kwargs['logits_to_keep'] = 1
            return orig(self, *args, **kwargs)
        return _ltk_fixed_fwd
    CsmDepthDecoderForCausalLM.forward = _make_ltk_fix(_orig_dd_fwd)
    CsmDepthDecoderForCausalLM._csm_logits_to_keep_patched = True
    print("Patched logits_to_keep: CsmDepthDecoderForCausalLM")

test_text = "具备可以交付的质量"
speaker_id = 0
inputs = processor(f"[{speaker_id}]{test_text}", add_special_tokens=True).to("cuda")

audio_values = model.generate(
    **inputs,
    max_new_tokens=125,
    output_audio=True,
)
audio = audio_values[0].to(torch.float32).cpu().numpy()

out_path = f"{OUTPUT_DIR}/test_inference.wav"
sf.write(out_path, audio, 24000)
print(f"Saved: {out_path}")
display(Audio(audio, rate=24000))